# Agentic Systems

**AI agents** are artificial intelligence systems that interacts with its environment (@fig-augmented-llm). We will be primarily interested in LLM-based agents that uses a core Large Language Model (LLM) functions as a computational engine for decision-making and planning. Since agentic systems typically implements a sequence of steps to achieve certain objectives, it is useful to distinguish these from more deterministic LLM-based systems (here called **workflows**):

- Workflows are predefined, static sequences where an LLM is invoked as a single step within a coded pipeline. The control flow and tool usage are **explicit** and **deterministic**. Here the generation is stochastic but the control flow around it is not. For example, a document translation and summarization pipeline.

- Agents are systems architected so the LLM itself dynamically reasons about and controls the sequence of actions. The model selects tools, evaluates outcomes, and iterates until a task is complete, making the path **non-deterministic** and **goal-oriented**.

Workflows provide predictability for well-structured tasks, while agents are suited for problems requiring flexibility and complex state management. A key engineering trade-off is that this autonomy typically [increases latency]{.underline} and [cost]{.underline} compared to a single LLM call, making architectural choice a critical design decision.


![**AI Agent building block.** Core LLM as engine for reasoning, planning, and tool calling. Agents follow goals and interact with their environment (external systems, such as other agents). This is not exactly new, the novelty lies in that AI agents does this in a highly performant and stochastic manner &mdash; very much like humans.](./img/augmented-llm.png){#fig-augmented-llm}

## Foundational papers

Four foundational papers that cover the capabilities of AI agents: **Chain-of-Thought Prompting** [@cot], **ReAct** [@ReAct2023], **Toolformer** [@toolformer], and **Generative Agents** [@genagents]. The rapid publication of ReAct, Toolformer, and Generative Agents within a few months of each other in early 2023 marks the period when LLMs transitioned from powerful stateless tools to the reasoning engines for autonomous, tool-using agents.

### Chain-of-Thought [@cot]

CoT is a technique where a non-reasoning model is prompted by few-shot examples of step-by-step reasoning steps that lead to the final output (@fig-cot). This resulted in SOTA performance (at the time) on the [GSM8K benchmark](https://github.com/openai/grade-school-math) consisting of grade school math word problems, from 18% to 57% solve rate for the PaLM 540B [*just* by prompting]{.underline} (surpassing even a fine-tuned GPT3 at 33%) &mdash; the authors benchmarked the technique with other tasks such as robot instruction and commonsense reasoning, likewise getting significant improvement.

These results were interpreted as LLM's having **emergent abilities** that can be *unlocked* if just asked to do so explicitly. It also opened the paradigm that fine-tuning was not necessary to get SOTA performance. Finally, since CoT only involves a change in prompt design, it was easily applicable to every LLM use-case.

![**Figure 1 of** [@cot]. Chain-of-thought prompting enables LLMs to tackle complex arithmetic, commonsense, and symbolic reasoning tasks just by a [simple trick]{.underline}. Chain-of-thought reasoning processes are highlighted.](img/cot.png){#fig-cot}

### ReAct [@ReAct2023]

CoT ("reasoning") was further augmented by **acting** (e.g. action
plan generation) in this paper. This paper 
explored the use of LLMs to generate both reasoning traces and task-specific actions in an interleaved manner resulting in greater synergy: reasoning traces help the model to play around action plans (e.g. handle exceptions), while actions allow it to interface with external sources (knowledge bases, Wikipedia) thereby gathering additional information (@fig-react-fig1). 

Performing actions resulted in improved **groundedness** and **trustworthiness** for the system (authors report 0% of ReAct failures are hallucinations, compared to 56% for CoT, instead 47% failures for ReAct are due to wrong reasoning traces incl. repetitive steps). In the paper, ReAct shows strong generalization to new task instances while "learning" solely from one to six in-context examples (i.e. no fine-tuning, manually compose ReAct-format trajectories), consistently outperforming only reasoning or only acting baselines across different domains. The authors also attempted to fine-tune a smaller model PaLM-8/62B via distilling[^distill] ReAct-format reasoning traces from PaLM-540B. The trained model outperformed all 540B pure-prompting methods.

![**Figure 1 of** [@ReAct2023]. The combination of reasoning and acting allows the agent to solve the task vs. reason-only and act-only approaches.](./img/react-paper.png){#fig-react-fig1}

[^distill]: The authors believe that finetuning with more human-written data may unleash the full potention of ReAct. Today, models are fine-tuned separately for reasoning and tool calling instead of at once.

### Toolformer [@toolformer] 

The next paper focused on **tool calling**. At this point, LLMs have been shown to be proficient in few-shot prompting. But it still struggles with basic tasks like arithmetic or [counting r's in "strawberry"](https://techcrunch.com/2024/08/27/why-ai-cant-spell-strawberry/). Moreover, it's knowledge is frozen after training cutoff, and as such is outdated by the time it gets to the user. All these issues are solved if the model is able to use tools. 

In the Toolformer paper, it is shown how a language model can be **trained** for **API calling** without fundamentally changing the model's architecture or training objective, and without a specialized dataset. The model learns to decide *which* API to call, *when* to call it, *what* arguments to pass, and *how* to best incorporate the results into the token generation process. The goal is to insert API calls into text generation. This were done in the ff. steps:

1. **Self-supervised data generation.** The key innovation of the paper is that the API calls are generated without needing human labelers. The model itself to generate potential API calls from text. The language model is exposed to few-shot examples of API calls inserted in [helpful places]{.underline} (the *when*). From Appendix A.2 of the paper, we see the prompts used. Here's one for the calculator tool (`Calculator` is one token):

    ```
    Your task is to add calls to a Calculator API to a piece of text. The calls sho
    uld help you get information required to complete the text. You can call the AP
    I by writing "❬Calculator(expression)❭" where "expression" is the expression to
     be computed. Here are some examples of API calls:
    
    Input:  The number in the next term is 18 + 12 x 3 = 54.
    Output: The number in the next term is 18 + 12 x 3 = ❬Calculator(18 + 12 *3)❭ 5
    4.

    Input:  A total of 252 qualifying matches were played, and 723 goals were score
    d (an average of 2.87 per match). This is three times less than the 2169 goals 
    last year.
    Output: A total of 252 qualifying matches were played, and 723 goals were score
    d (an average of ❬Calculator(723 / 252)❭ 2.87 per match). This is twenty goals 
    more than the ❬Calculator(723 - 20)❭ 703 goals last year.
    
    Input: {x}
    Output:
    ```

    Then, for an input sequence $\mathbf{x}$ we iterate over positions $i$ in the text and save those where 
    the model conditioned on $[\Phi(\mathbf{x}_{<i}), \mathbf{x}_{<i}, \boldsymbol{\langle}]$ where $\Phi(\mathbf{x}_{<i})$ is the accompanying tool call prompt (above) assigns a high probability of an API call as 
    next token[^sampling_threshold]. Next, for each such position we get a number of API args $\mathbf{c}^1_i, \ldots, \mathbf{c}^m_i$ (one tool with different arguments) by generating text until the model outputs $\boldsymbol{\rangle}$ as EOS token (examples that do not generate this are discarded). Note that [no actual execution]{.underline} is performed so far. We only want a sample of positions along with API args for that position for each input $\mathbf{x}.$[^phase1]

    ![**Examples of API calls and responses.** The tools are designed to take natural language as input. The tool specification influence the design of few-shot prompts $\Phi.$ See paper for details (e.g. the MT tool serves [NLLB-600M](https://huggingface.co/facebook/nllb-200-distilled-600M) a multi-lingual machine translation model for 200 languages with [fastText](https://github.com/facebookresearch/fastText) to identify the source language).](./img/toolformer-apicalls.png)

    In the next phase, we perform the executions of the sampled API args $\mathbf{c}^1_i, \ldots, \mathbf{c}^m_i$. For $l = 1, \ldots, m$, the response needs to be a single text sequence $\mathbf{r}_i^l$ (this includes error responses). Finally, we filter out the sample API calls by evaluating to see if the tokens of $\mathbf{x}$ after $i$ is assigned a significantly lower loss (vs. some error threshold[^api_threshold] $\tau_f$) vs. the generation [without]{.underline} tool call[^actual]. 
    The API calls with their execution results are then interleaved in the text to get $\mathbf{x}^* = [\mathbf{x}_{<i}, \boldsymbol{\langle}\mathbf{c}_i \rightarrow \mathbf{r}_i \boldsymbol{\rangle}, \mathbf{x}_{\geq i} ].$
    Doing this process of candidate generation, execution, and filtering (@fig-toolformer) to every text in the original dataset yields the final, curated dataset used for fine-tuning. 
    Some examples in the fine-tuning dataset[^ft-format]:

    ```
    The name derives from “la tortuga”, the Spanish word for ❬MT(“tortuga”) → turtle❭ turtle.
    Out of 1400 participants, 400 (or ❬Calculator(400 / 1400) → 0.29❭ 29%) passed the test. 
    ```
    
2. **Standard fine-tuning & modified inference.** The language model is then trained on the augmented tool-use dataset using standard techniques. Note that the output of the API calls are already in the dataset so we don't have to perform any execution at this stage. However, during inference the engine is designed such that whenever the token `→` is detected, the decoding is paused to execute the API call and the generation continues with the result and the EOS `❭` added to the context. The authors used greedy decoding with `❬` selected if it's in the top-*k* of candidate tokens to encourage tool-calling, with limitations on the amount of tool calls per input. The model is expected to perform tool-calling in a **zero-shot** setup (i.e. no examples of tool-calling are needed in its prompt.)

![**Figure 2 of** [@toolformer]. A language modeling dataset is augmented with API calls. Instead of just predicting "the Steel City", an API call is created as an in-between target.](./img/toolformer.png){#fig-toolformer}

[^sampling_threshold]: The top-$k$ positions in the text $\mathbf{x}$ with next-token probability of an API call that exceeds $\tau_s = 0.05$ or 5%. For large datasets, $k=5$ and $m=5$ were used. For smaller, this is increased to $k=20$, $m=10$, and $\tau_s = 0.$

[^phase1]: This phase focuses on gathering candidate data that can potentially teach a model on 
*when* to call and API and *which* API arguments to call. Not so much how the actual return values of the 
call affects future generation.

[^api_threshold]: The size of the generated dataset decrease rapidly as $\tau_f$ increases (Table 2).

[^actual]: Suppose the API call happens at index $i$ in a sequence $\mathbf{x}$, the model is conditioned
on $[\mathbf{z}, \mathbf{x}_{<i}]$ and a weighted cross-entropy of the ground truth sequence $\mathbf{x}_{\geq i}$ is calculated, with the prefix $\mathbf{z}$ being either (1) FULL `❬f(args) → r❭` where $\mathbf{c}_i =$ `f(args)` and $\mathbf{r}_i =$ `r`, (2) MASKED RESULT `❬f(args) → ❭` and (3) NO API call (i.e. the usual generation). A sample API call is kept only if the weighted cross-entropy of $\mathbf{x}_{\geq i}$ (1) is the minimum compared to (2) and (3) by at least $\tau_f.$ Machine Translation (`MT`) tool example:
```
(1) [❬MT(“tortuga”) → turtle❭ The name derives from “la tortuga”, the Spanish word for] turtle.
(2) [❬MT(“tortuga”) → ❭ The name derives from “la tortuga”, the Spanish word for] turtle.
(3) [The name derives from “la tortuga”, the Spanish word for] turtle.
```
This intuitively means that the act of calling a particular API is not only useful, but its result is also useful. Here the API call $\mathbf{z}$ is added as a prefix in getting $p_\textsf{model}(\mathbf{x}_{\geq i} \mid [\mathbf{z}, \mathbf{x}_{<i}])$ is so that the complete sequence $[\mathbf{z}, \mathbf{x}_{<i}, \mathbf{x}_{\geq i}]$ remains in-distribution w.r.t. the pretraining data.
<br><br>
Finally, the cross-entropy weights $w_t \geq 0$ where $t = j - i$ decay for increasing $t.$ 
The authors used $w_t = \max(0, 1 - 0.2 \cdot t)$ which means only 6 next tokens contribute to the loss. 
These are normalized over the sequence, i.e. the actual weights
used are $\tilde{w}_t = w_t / \sum_{t^\prime} w_{t^\prime}.$
Using decaying weights ensures that API calls happen close to where
the information provided by the API is actually helpful for the model. 

[^ft-format]: The format is exactly how the API calls during the generation process are performed. In the actual paper, `[`, `]`, and `->` are used so that everything works without modifying the existing vocabulary. 

The self-supervised data generation process relies on base model already possessing a rich, implicit knowledge of tools and their purposes from its pre-training (humans use tools intuitively, and tool use follows causal and physical rules, all of these reflect in recorded language data, so it's likely that LLMs also learn it from the statistics of the data). Assuming this is true, what the LLM then lacks is the 'grammar' for tool calls[^grammar_tool_call]. The fine-tuning phase solved this by forging a new, high-probability pathway into its output generation (i.e. using the tool call token $\text{❬}$ and forming proper arguments) in contexts where information has to be identified from external sources (@fig-toolformer-evals).

![**Performance of Toolformer in downstream tasks.** Toolformer was evaluated on a variety of downstream tasks. A zero-shot setup is used where no tool-calling examples were provided in the prompt. Greedy decoding was used with an API call triggered when $\text{❬}$ is found in the top-10 next token candidates. Toolformer with [tool-calling disabled]{.underline} was also evaluated to ensure that the base performance did not degrade (i.e. forcing $p(\text{❬}) = 0$). GPT-J + CC refers to [GPT-J](https://en.wikipedia.org/wiki/GPT-J) (6B) fine-tuned on a subset of [Common Crawl](https://en.wikipedia.org/wiki/Common_Crawl), and Toolformer is the same GPT-J fine-tuned on CC$^*$ (i.e. Common Crawl augmented with API calling discussed). Interestingly, Toolformer with API calling disabled performs better than GPT-J + CC &mdash; it is surmised that being exposed more API calls and their results during training improved its own reasoning abilities[^pure-lm-perf]. Effect of tool calling is also emphasized by beating GPT-3 (175B) a significantly larger model in these tasks.](./img/toolformer-evals.png){#fig-toolformer-evals}

[^grammar_tool_call]: Toolformer translated the tool-use problem into a pure text completion problem, and showed it was possible. Without having read the paper, I would think an RL-based approach would be required to successfully solve this. However, from the empirical analysis of scaling laws in the paper (Figure 4), it was shown that Toolformer (GPT-J fine-tune) only beats Toolformer (disabled &mdash; i.e. forcing $p(\text{❬}) = 0$) past **775M parameters**, implying that tool-calling capabilities started to emerge once language ability was sufficiently high. The scaling laws also show that the gap between Toolformer and Toolformer (disabled) does not decrease past the parameter count where the model learns tool-calling.

[^pure-lm-perf]: How about performance on pure language modeling (e.g. "Write me a story")? **Table 8** of the paper shows base language modeling performance of the models in terms of perplexity. It is shown that Toolformer (disabled) (+ CC$^*$) did not degrade relative to GPT-J + CC which is the valid comparison. It did degrade in [WikiText](https://huggingface.co/datasets/Salesforce/wikitext) which is fine since GPT-J + CC also did, so this may be just due to distribution shift.

### Generative agents [@genagents]

Finally, the last paper *Generative Agents: Interactive Simulacra of Human Behavior* dives into the ability of agents to simulate believable human behavior. This highlights the ability of LLM-based agents to interact with their **environment** which includes other agents. Hence, providing an example of a working multi-agent architecture. The agents in this paper are characterized as being able to **remember**, **reflect**, and **plan** based on growing memory and casading social dynamics. In the paper, a sandbox environment are filled with 25 LLM-based agents that are shown to demonstrate believable human behavior (@fig-genagents). 

![**Figure 1 of** [@genagents]. Generative agents socialize, chill out, talk about politics & news, and perform routines. Moreover they have been observed to perform longer-term planning and coordinate with other agents in their plans.](./img/genagents.png){#fig-genagents}

**Generative agents.** Each agent is [seeded]{.underline} with a paragraph in natural language that depict their identity (including their occupation and relationship with other agents). Each `;`-delimited phrase is entered into the agent’s **initial memory** as memories at the start of the simulation (later we will discuss the memory stream in more detail). 

Generative agents operate in
an action loop where, at each time step, they perceive the world
around them and perform actions, or talk to other agents. 
Consider the agent John Lin. John Lin
interacts with their world in several key ways:

1. **Action.** At each time step, an agent outputs a natural language statement describing its current action ("John Lin is researching the local mayor election"). This includes interaction with their environment (e.g. "closet is being used to select clothes for the day"). Agents can also move between different locations in the map. 

2. **Observation.** Agents can record the state of their local environment in their memory ("refrigerator is idle"). This includes other agents: "Adam Smith is discussing the topic with the other participants". 

3. **Dialogue.** Agents converse as they interact with each other. Agents' dialogue are generated by conditioning 
on their memory of each other and the current summary status of the agent, as well as their observation of the 
other agent whom they want to engage in a conversation with:

    ```
    [Agent's Summary Description:]
    It is February 13, 2023, 4:56 pm. 
    John Lin's status: John is back home early from work. 

    [Observation:] 
    John saw Eddy taking a short walk
    around his workplace.

    [Summary of relevant context from John's memory:]
    Eddy Lin is John's Lin's son. Eddy Lin has been
    working on a music composition for his class. Eddy
    Lin likes to walk around the garden when he is
    thinking about or listening to music.
    John is asking Eddy about his music composition
    project. 

    What would he say to Eddy?
    ```

The continuation of this dialogue is generated using
the same mechanism until one of the two agents decides to end the
dialogue. Below (in the *reacting* discussion) it is discussed in more detail how inter-agent dialogues are triggered.

:::{.callout-tip}
## Emergent behavior
These interactions result in emergent behavior such as [information diffusion]{.underline} and [coordination]{.underline} (planning an event involving multiple agents helping each other out and arriving on time at a specific location).
:::


**Memory and retrieval.** Everything that an agent experiences is recorded and reasoned over as a natural language description, 
allowing the architecture to leverage a LLM. 
Generative agents take their current environment and past experiences as [input]{.underline} and generate behavior as output.
Underlying this behavior is a novel agent architecture that combines a large language model with mechanisms
for synthesizing and retrieving relevant information to condition
the language model’s output.

- **Memory stream.** This is a list of 
memory objects for each agent that contains a timestamp created, timestamp of last access (which defaults to created), 
and a natural language description. 
Observations, i.e. event directly perceived by agent (e.g. behavior of self, other, non-agent objects)
are stored in memory. Conversation between agents are summarized in the memory stream.

- **Retrieval function.** Because generative agents produce large streams of events and memories
that must be retained, a core challenge is ensuring that the most relevant pieces of the agent's memory are
retrieved and synthesized when needed. For a query $Q$, we retrieve memory items $m$ based on the score:
    $$\text{score}(m, Q, t) = \alpha_1 \cdot 0.995^{\Delta t_m} + \alpha_2 \cdot \text{I}(m) + \alpha_3 \cdot (\hat{\mathbf{v}}_Q \cdot \hat{\mathbf{v}}_m)$$
    where $\alpha_1 + \alpha_2 + \alpha_3 = 1.$
    Here $t$ is the current time, so that the first term is a *recency* score that decays exponentially. 
    For example, events that occured on the same day are more likely to be retrieved, all things being equal, 
    compared to events that occured on the previous week. Next, the *importance* $\,\text{I}(m)$ is determined by the LLM itself
    by prompting:

    ```
    On the scale of 1 to 10, where 1 is purely mundane (e.g., brushing teeth, 
    making bed) and 10 is extremely poignant (e.g., a break up, college accep
    tance), rate the likely poignancy of the following piece of memory.
    
    Memory: buying groceries at The Willows Market and Pharmacy
    Rating: <fill in>
    ```

    Finally, we calculate relevance as the cosine similarity between the memory's 
    embedding vector $\hat{\mathbf{v}}_m$ and the query memory's embedding vector $\hat{\mathbf{v}}_Q$ using the LLM. Here unit vectors are 
    used, so we just dot these to get cosine similarity. The top-ranked memories that fit within
    the language model’s context window are included in the prompt upon which the retrieval function is used
    to fill with memory items.


![**Figure 6 of ** [@genagents]. The memory stream consists of a large number of observations that are relevant and irrelevant to the agent's current situation. Retrieval identifies a subset of these observations that should be passed to the language model to condition its
response to the situation.](./img/genagents-memory-retrieval.png)


**Reflection.** 
Generative agents, when equipped with only raw observational memory, struggle to generalize or make inferences.
Reflections are introduced as a second type of memory (e.g. the retrieval function also returns reflections). 
Reflections are higher-level, more abstract thoughts
generated by the agent. Because they are a type of memory, they
are included alongside other observations when retrieval occurs. In the paper, this triggers for an agent
when the sum of importance scores exceed 150. This is done as follows:

1. **Three points of reflection.** Query the 100 most recent memory. Prompt the language model: "Given only the information above, what are 3 most salient high-level questions we can answer about the subjects in the statements?". This method directly extracts specific lines of inquiry rather than producing a general summary.

2. **Reflection proper.** For one question $Q$ at a time, retrieve memories using the retrieval function $f(Q).$ Prompt the language model to extract insights:
    
    ```text
    Statements about Klaus Mueller
    1. Klaus Mueller is writing a research paper
    2. Klaus Mueller enjoys reading a book
    on gentrification
    3. Klaus Mueller is conversing with Ayesha Khan
    about exercising [...]

    What 5 high-level insights can you infer from the above statements? 
    (example format: insight (because of 1, 5, 3))
    ```

3. **Extend reflection tree.** Store these in the memory stream and cite the particular records that served as evidence for the insights. Since reflections are memories, agents can reflect on past reflections.
As a result, agents generate trees of reflections (@fig-reflections-tree): the [leaf nodes]{.underline} of
the tree represent the base observations, and the non-leaf nodes
represent [thoughts]{.underline} that become more abstract and higher-level the
higher up the tree they are. Hence, we can assign a **depth** value to each memory.

![**Figure 7 of ** [@genagents] A reflection tree for Klaus Mueller. The agent’s observations of the world, represented in the leaf nodes, are recursively synthesized to derive Klaus’s self-notion that he is highly dedicated to his research.](./img/reflections-tree.png){#fig-reflections-tree}


**Planning and reacting.** Agents need to plan over a longer time horizon to ensure that their sequence
of actions is coherent and believable. A **plan** includes a location, a starting time, and a duration.
Like reflections, plans are [stored in the memory stream]{.underline} and are included in the retrieval process. This
allows the agent to consider observations, reflections, and plans all together when deciding how to behave. 

First, at the start of each day, the language model is prompted with the agent's summary description (e.g. name, traits, and a summary of their recent experiences) and a summary of their previous day. A general plan for the day's agenda is obtained (7-8 items). These are then recursively decomposed to create finer-grained actions, first into hour-long chunks of actions, and then to 5-15 minute chunks (@fig-genagents-plan).

Agents may [change plans]{.underline} mid-stream if needed given observations from their environment. We prompt the language model with these observations to decide whether the agent should continue with their existing plan, or **react**. The prompt template is given by:
```text
[Agent's Summary Description:] 
{timestamp,agent_current_state}

[Observation:] 
{observation} 

[Summary of relevant context from John’s memory:] 
{context_summary}

Should John react to the observation, and if so, 
what would be an appropriate reaction?
``` 
Let `O` be the observer and `E` be the observed entity. The context is generated through two prompts that retrieve memories by summarizing the combined output of the two queries with the retrieval function: `"What is [O]'s relationship with [E]?"` and `"[E] is [status_of_E]"`.
Note that these include plans, since plans are also in the memory stream.
We then regenerate the agent's existing plan starting from the time when the reaction takes place. Finally, if the output action indicates an interaction between agents, we generate their dialogue.

![**Planned tasks of Yuriko Yamamoto.** Current status: Yuriko Yamamoto is working on a tax compliance project for a local business. She is also taking classes to stay up to date on new tax laws. Yuriko is also curious about who will be running for the local mayor election next month.](./img/genagents-plan.png){#fig-genagents-plan}

:::{.callout-note}
**Generative Agents** [@genagents] showed that LLMs can simulate believable behavior by having (1) a [seeded agent identity]{.underline}, and (2) a [core memory module]{.underline} that consist of observations, reflections, and current plans. Since memories are noisy and will quickly fill LLM context, having a [retrieval function]{.underline} is crucial given the query that applies to the current situation. 
This is augmented by having [reflections]{.underline} which are abstract observations that compress previous events and that were recollected based on recent events. 
See Appendix B of the paper where agents are interviewed (e.g. *Give an introduction of yourself*) which can 
be answered using the retrieval function.
Finally, allowing LLMs to interact results in [emergent phenomenon]{.underline} (i.e. occurences that not explicitly built into the system).
:::

## Contents

### General concepts

- [Augmented LLMs as Agents](/topics/agents/01.html)

### Four Agentic Patterns

In the following notebooks, we look at four agentic patterns: (1) **reflection**, (2) **tool use**, (3) **planning** / **ReAct** [@ReAct2023], (4) **multi-agent** pattern. This series builds on [this repo](https://github.com/neural-maze/agentic-patterns-course?tab=readme-ov-file) by [`@neural-maze`](https://theneuralmaze.substack.com/). The contents are summarized in the following table:



| Pattern              | Key Components                              | Description                                                                 |
|----------------------|---------------------------------------------|-----------------------------------------------------------|
| **Reflection**       | Generate ⟳ Reflect                          | Iterative cycle where the model produces outputs, reflects on them, and improves future generations. |
| **Tool Use**         | Tools to external systems | The model selects and applies external tools to extend its capabilities.   |
| **Planning (ReAct)** | Think ↔ Act ↔ Obs | Combines reasoning ("thought") with actions and feedback from observations in a loop. |
| **Multi-Agent**      | Agent 1 → Agent 2 → Agent 3       | Multiple agents collaborate or sequence their actions to solve complex tasks. |



- [Pattern 1: Reflection](/topics/agents/02.html)
- [Pattern 2: Tool Use](/topics/agents/03.html)
- [Pattern 3: Planning / ReAct](/topics/agents/04.html)
- [Pattern 4: Multi-Agent](/topics/agents/05.html)

### Projects

- 

## Utility functions

### Chat completion

Usual boilerplate when dealing with chat completions.

In [1]:
class Role:
    USER = "user"
    TOOL = "tool"
    SYSTEM = "system"
    ASSISTANT = "assistant"

    @classmethod
    def get_valid_roles(cls) -> set:
        """Automatically detect all uppercase constant roles"""
        return {value for name, value in vars(cls).items() 
                if name.isupper() and isinstance(value, str)}
    
    @classmethod
    def validate(cls, role: str) -> str:
        valid_roles = cls.get_valid_roles()
        if role not in valid_roles:
            raise ValueError(f"Invalid role: {role}")
        return role


def completions_create(client, messages: list, model: str) -> str:
    """Return generated string from model based on messages."""
    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def message_dict(prompt: str, role: str, tag: str = "") -> dict:
    """Return a message dictionary for the chat completions API."""
    role = Role.validate(role)
    prompt = f"<{tag}>{prompt}</{tag}>" if tag else prompt
    return {"role": role, "content": prompt}


# example
print(Role.get_valid_roles())

try:
    print(message_dict(role="user", prompt="Hello", tag="greeting"))
    print(message_dict(role="test", prompt="Hello", tag="greeting"))
except ValueError as e:
    print(e)

{'system', 'assistant', 'tool', 'user'}
{'role': 'user', 'content': '<greeting>Hello</greeting>'}
Invalid role: test


Implementing chat history class to abstract appending messages with limit to naively prevent ["context overflow"](https://aws.amazon.com/blogs/security/context-window-overflow-breaking-the-barrier/). We have the parameter `fixed_n` (default `1`) to preserve first `n` message since it is often important (e.g. `n=1` for the system prompt).

In [2]:
from typing import Optional


class ChatHistory(list):
    def __init__(self, 
        system_prompt: Optional[str] = None, 
        messages: Optional[list] = None,
        max_len: int = -1, 
        fixed_n: int = 1
    ):
        """Fixed message list with a optional total length and number of fixed initial messages."""
        messages = [] if messages is None else messages
        super().__init__(messages)
        assert max_len > 1 or max_len == -1, "max_len must be -1 (no limit) or > 1"
        assert bool(system_prompt) + bool(messages) <= 1
        self.fixed_n = fixed_n
        self.max_len = max_len
        if system_prompt:
            self.update(prompt=system_prompt, role=Role.SYSTEM)
        
    def append(self, chat: dict):
        if len(self) == self.max_len:
            self.pop(self.fixed_n)    # i.e. keep 0, 1, ..., n-1 (first n)
        chat["role"] = Role.validate(chat["role"])
        super().append(chat)

    def update(self, prompt: str, role: str, tag: str = ""):
        """Append a message to the chat history."""
        self.append(message_dict(prompt=prompt, role=role, tag=tag))


chat_history = ChatHistory(
    system_prompt="you are a goldfish", max_len=3, fixed_n=1
)
chat_history.update("1", "user")
chat_history.update("2", "user")
chat_history.update("3", "user", tag="last")
chat_history

[{'role': 'system', 'content': 'you are a goldfish'},
 {'role': 'user', 'content': '2'},
 {'role': 'user', 'content': '<last>3</last>'}]

In [3]:
for cmd in [
    lambda: chat_history.append({"role": "test", "content": "test"}),
    lambda: chat_history.update(role="test", prompt="test")
]:
    try:
        cmd()
    except ValueError as e:
        print(e)

Invalid role: test
Invalid role: test


### Tag extraction

The following utilities will be used to extract content from tags (e.g. `<thought>`, `<response>`, etc).

In [4]:
import re
from dataclasses import dataclass


@dataclass
class TagContentResult:
    content: list[str]
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """
    Extracts all content enclosed by specified tags, 
    e.g. <thought>, <response>, etc.
    Parameters:
        text (str): The input string containing multiple potential tags
        tag  (str): The name of the tag to search for
    """
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    return TagContentResult(
        content=[content.strip() for content in matched_contents],
        found=bool(matched_contents),
    )

message = """
<thought>This is a thought.</thought> 
<response>This is a response.</response>
<thought>This is another thought.</thought> 
"""
print(extract_tag_content(message, "thought"))
print(extract_tag_content(message, "response"))
print(extract_tag_content(message, "tool"))

TagContentResult(content=['This is a thought.', 'This is another thought.'], found=True)
TagContentResult(content=['This is a response.'], found=True)
TagContentResult(content=[], found=False)
